# Importing Libraries and Functions

In [1]:
# Import libraries
import kagglehub
import pandas as pd
import numpy as np

import sys
from pathlib import Path

# Import functions from src/features.py
sys.path.append(
    str(Path().resolve().parent)
)

from src.features import (
    get_match_result,
    goal_diff,
    add_position_group,
    team_rating
)

/Users/edenaong/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/edenaong/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Load Data

In [2]:
# Load Cleaned Datasets
results = pd.read_csv("../data/processed/results_clean.csv")
wc2026_draw = pd.read_csv("../data/processed/wc2026_draw_clean.csv")
players = pd.read_csv("../data/processed/players_clean.csv")
elo_ratings = pd.read_csv("../data/processed/elo_clean.csv")

In [3]:
results["date"] = pd.to_datetime(results["date"])
elo_ratings["rank_date"] = pd.to_datetime(elo_ratings["rank_date"])

# Features
## Team Ratings 
### 1. Squad Rating
The average rating of the 26 person squad (the maximum number of players in a squad) for each country.

In [ ]:
squad_requirements = { # General squad requirements
    "GK": 3,
    "DEF": 9,
    "MID": 9,
    "FWD": 5
}

# Generating squad ratings for all countries based on top players in each position group, with fill rating for missing players
squad_ratings = team_rating(players, requirements=squad_requirements, rating_name="squad_rating")
squad_ratings.sort_values(["avg_rating", "team_size"], ascending=[False, False])

,country,avg_rating,team_size,missing_players
0,France,84.230769,26,0
3,England,84.076923,26,0
1,Spain,83.730769,26,0
4,Brazil,83.615385,26,0
8,Germany,83.576923,26,0
6,Argentina,83.038462,26,0
10,Portugal,83.000000,26,0
7,Netherlands,81.730769,26,0
5,Belgium,79.884615,26,0
11,Uruguay,78.269231,26,0


### 2. Top 11 Rating

The average rating of the top 11 players in the squad for each country.

In [ ]:
top11_requirements = { # 4-3-3 formation as estimate for starting 11 requirements
    "GK": 1,
    "DEF": 4,
    "MID": 3,
    "FWD": 3
}

# Generating top 11 ratings for all countries based on top players in each position group, with fill rating for missing players
top11_ratings = team_rating(players, requirements=top11_requirements, rating_name="top11_rating")
top11_ratings.sort_values(["avg_rating", "team_size"], ascending=[False, False])

,country,avg_rating,team_size,missing_players
0,France,86.272727,11,0
3,England,86.272727,11,0
4,Brazil,86.181818,11,0
8,Germany,85.636364,11,0
10,Portugal,85.454545,11,0
6,Argentina,85.272727,11,0
1,Spain,84.909091,11,0
7,Netherlands,83.272727,11,0
5,Belgium,82.363636,11,0
11,Uruguay,81.545455,11,0


## Elo Features

### Preparing the data for merging

In [6]:
# Standardising country names between results and elo datasets
country_map = {
    "Brunei Darussalam": "Brunei",
    "Chinese Taipei": "Taiwan",
    "Hong Kong, China": "Hong Kong",
    "Korea DPR": "North Korea",
    "Kyrgyz Republic": "Kyrgyzstan",
    "St Kitts and Nevis": "Saint Kitts and Nevis",
    "St Lucia": "Saint Lucia",
    "St Vincent and the Grenadines": "Saint Vincent and the Grenadines",
    "The Gambia": "Gambia",
    "US Virgin Islands": "United States Virgin Islands"
}

# Apply country name mapping to elo dataset
elo_ratings["country_full"] = elo_ratings["country_full"].replace(country_map)

In [7]:
# Filter results to only include matches between countries present in the Elo ratings dataset as some countries are obsolete and not present in elo dataset
valid_countries = set(elo_ratings["country_full"])
results = results[results["home_team"].isin(valid_countries) & results["away_team"].isin(valid_countries)]

In [8]:
# Checking for any remaining discrepancies in country names between results and elo datasets
results_countries = set(results["home_team"]).union(
    set(results["away_team"])
)

elo_countries = set(elo_ratings["country_full"])

print("In results but not elo:")
print(sorted(results_countries - elo_countries))

print("\nIn elo but not results:")
print(sorted(elo_countries - results_countries))

In results but not elo:
[]

In elo but not results:
['Netherlands Antilles', 'Serbia and Montenegro', 'Zaire']


In [9]:
# Sorting by date for good measure, though it should already be sorted
elo_ratings = elo_ratings.sort_values("rank_date")
results = results.sort_values("date")

# Home team Elo ratings
home_elo = elo_ratings.rename(columns={
    "country_full": "home_team",
    "total_points": "home_elo"
})
# Merge home team Elo ratings with results
results = pd.merge_asof(
    results.sort_values("date"),
    home_elo.sort_values("rank_date"),
    left_on="date",
    right_on="rank_date",
    by="home_team",
    direction="backward"
)

# Away team Elo ratings
away_elo = elo_ratings.rename(columns={
    "country_full": "away_team",
    "total_points": "away_elo"
})

# Merge away team Elo ratings with results
results = pd.merge_asof(
    results.sort_values("date"),
    away_elo.sort_values("rank_date"),
    left_on="date",
    right_on="rank_date",
    by="away_team",
    direction="backward"
)

### Creating Features

1. elo_diff feature

The difference in elo rating between home and away teams at the time of each match.

In [10]:
results["elo_diff"] = results["home_elo"] - results["away_elo"]

2. home_higher_elo feature

If the home_team has a higher elo rating than the away team, the value is 1. Else, 0.

In [11]:
results["home_higher_elo"] = (results["home_elo"] > results["away_elo"]).astype(int)

In [12]:
results[["date", "home_team", "away_team", "home_elo", "away_elo", "elo_diff", "home_higher_elo"]].sample(10)

,date,home_team,away_team,home_elo,away_elo,elo_diff,home_higher_elo
41407,2021-07-13,South Africa,Lesotho,1325.64,1069.91,255.73,1
44569,2024-09-07,Faroe Islands,North Macedonia,1093.70,1348.63,-254.93,0
5104,1962-05-12,Brazil,Wales,NaN,NaN,NaN,0
42856,2022-11-26,Argentina,Mexico,1773.88,1644.89,128.99,1
27975,2006-01-27,Nigeria,Zimbabwe,692.00,599.00,93.00,1
5201,1962-09-16,Congo,Gabon,NaN,NaN,NaN,0
19164,1995-09-06,Czech Republic,Norway,47.00,59.00,-12.00,0
39514,2019-01-10,Jordan,Syria,1196.00,1322.00,-126.00,0
33069,2011-10-07,Greece,Croatia,1000.00,1057.00,-57.00,0
21385,1998-06-24,Andorra,Azerbaijan,2.00,21.00,-19.00,0


## Recent Form Features